# 02 — GSE144735 Preprocessing (external validation)

Cell-intrinsic preprocessing **identical to notebook 01**, in the same order:
QC → gene filter → artifact removal → Scrublet → normalize + log1p.

**Revision notes (v2):**
1. **Reproducibility.** The previous version was executed out of order (QC ran twice,
   exec_count 10 after Scrublet's 7), so the saved object could not be reproduced by a
   top-to-bottom run. This notebook must be executed with *Restart Kernel → Run All*.
2. **Identical QC configuration** to discovery: no mitochondrial rule (source matrix
   already capped at 20% MT), `pct_counts_in_top_20_genes` computed **within cell type**.
   A composition audit is printed so filtering cannot silently bias the contrast.
3. **Artifact filter applied here too.** Notebook 01 ran Scrublet *after* artifact removal;
   doing the same here makes doublet detection genuinely identical across cohorts.
4. **Border cells are NOT collapsed here.** `Class_original` keeps Normal/Border/Tumor and
   `Class` holds the Border→Tumor convention. Notebook 04 can then run the primary
   analysis and the Border sensitivity analysis without re-running preprocessing.
   Border is 53% of the validation tumour class, so this must be auditable.
5. **No HVG selection.** The feature space is fixed by the discovery HVG list; notebook 04
   uses the *intersection* (no zero-filling). Full gene space is saved.

In [1]:
# ============================================================
# 0. Imports, determinism, config (mirrors notebook 01)
# ============================================================
import os, warnings, random, re
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import median_abs_deviation

import anndata
anndata.settings.allow_write_nullable_strings = True

RANDOM_STATE = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
sc.settings.verbosity = 1

# ---------- QC parameters: MUST match notebook 01 ----------
NMADS_COUNTS = 5
NMADS_TOP20  = 5
MT_RULE      = "off"          # source matrix already capped at 20% MT

# ---------- paths ----------
COUNT_FILE      = "GSE144735_count_matrix.txt.gz"
ANNOTATION_FILE = "GSE144735_annotation.txt.gz"
OUT_FILE        = "GSE144735_DL_ready.h5ad"
DISC_DIR        = "gse132465_preprocessing_outputs_before_Xm"   # notebook 01 OUT_DIR
HVG_REF         = os.path.join(DISC_DIR, "GSE132465_HVG3000_gene_list.csv")

class_col    = "Class"
celltype_col = "Cell_type"
sample_col   = "Sample"
patient_col  = "Patient"

print("Discovery reference dir:", DISC_DIR)
sc.logging.print_versions()

Discovery reference dir: gse132465_preprocessing_outputs_before_Xm


Package,Version
numpy,2.5.2
pandas,2.3.3
scanpy,1.12.3
scipy,1.18.1
anndata,0.13.3.post0
Component,Info
Python,"3.12.3 (main, Jun 19 2026, 12:46:00) [GCC 13.3.0]"
OS,Linux-7.0.0-30-generic-x86_64-with-glibc2.39
CPU,"24/24 logical CPU cores, x86_64"
GPU,"ID: 0, NVIDIA GeForce RTX 5090, Driver: 595.84, Memory: 32607 MiB"


In [2]:
# ============================================================
# 1. Load count matrix + annotation
# ============================================================
print("Loading count matrix...")
val = sc.read_text(COUNT_FILE, delimiter="\t").T
print(f"Raw: {val.n_obs:,} cells x {val.n_vars:,} genes")

meta = pd.read_csv(ANNOTATION_FILE, sep="\t", index_col=0)
print("Annotation shape:", meta.shape, "| columns:", list(meta.columns))

common = val.obs_names.intersection(meta.index)
print(f"Common cells: {len(common):,}")
if len(common) == 0:
    raise ValueError("No matching cell IDs between count matrix and annotation.")

val = val[common].copy()
val.obs = val.obs.join(meta.loc[val.obs_names])
val.layers["counts"] = val.X.copy()
print(val)

Loading count matrix...
Raw: 27,414 cells x 33,694 genes
Annotation shape: (27414, 5) | columns: ['Patient', 'Class', 'Sample', 'Cell_type', 'Cell_subtype']
Common cells: 27,414
AnnData object with n_obs × n_vars = 27414 × 33694
    obs: 'Patient', 'Class', 'Sample', 'Cell_type', 'Cell_subtype'
    layers: None (.X), 'counts'


In [3]:
# ============================================================
# 2. Class handling: keep Border auditable
# ============================================================
if patient_col not in val.obs.columns:
    print(f"WARNING: '{patient_col}' absent; using '{sample_col}' as group.")
    patient_col = sample_col
for c in [class_col, celltype_col, sample_col]:
    if c not in val.obs.columns:
        raise ValueError(f"Missing required column '{c}'. Available: {list(val.obs.columns)}")

orig = val.obs[class_col].astype(str)
print("Original Class values:", sorted(orig.unique()))
print(orig.value_counts())

# Keep the three-level tissue label AND the binary modelling label side by side.
val.obs["Class_original"] = pd.Categorical(orig, categories=["Normal", "Border", "Tumor"])
val.obs[class_col] = pd.Categorical(orig.replace({"Border": "Tumor"}).values,
                                    categories=["Normal", "Tumor"])

n_border = int((orig == "Border").sum())
n_tumor  = int((val.obs[class_col] == "Tumor").sum())
print(f"\nBorder -> Tumor: {n_border:,} cells "
      f"({100 * n_border / max(n_tumor, 1):.1f}% of the Tumor class)")
print(val.obs[class_col].value_counts())

for col in [celltype_col, sample_col, patient_col]:
    val.obs[col] = val.obs[col].astype("category")

comp_before = pd.crosstab(val.obs[celltype_col], val.obs["Class_original"])
print("\nPRE-QC cell type x tissue composition:")
print(comp_before)
print("\nSamples:", val.obs[sample_col].nunique(), "| Patients:", val.obs[patient_col].nunique())

Original Class values: ['Border', 'Normal', 'Tumor']
Class
Normal    9736
Border    9424
Tumor     8254
Name: count, dtype: int64

Border -> Tumor: 9,424 cells (53.3% of the Tumor class)
Class
Tumor     17678
Normal     9736
Name: count, dtype: int64

PRE-QC cell type x tissue composition:
Class_original    Normal  Border  Tumor
Cell_type                              
B cells             2604    1521    777
Epithelial cells    1144    2812   2212
Mast cells            62     108     78
Myeloids             823     924    929
Stromal cells       3564    1765   2321
T cells             1539    2294   1937

Samples: 18 | Patients: 6


In [4]:
# ============================================================
# 3. QC metrics (identical definitions to notebook 01)
# ============================================================
val.var["mt"]   = val.var_names.str.startswith("MT-")
val.var["ribo"] = val.var_names.str.startswith(("RPS", "RPL"))
val.var["hb"]   = val.var_names.str.contains(r"^HB[^P]", regex=True)

sc.pp.calculate_qc_metrics(val, qc_vars=["mt", "ribo", "hb"],
                           percent_top=[20], log1p=True, inplace=True)

print(val.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt",
               "pct_counts_ribo", "pct_counts_hb",
               "pct_counts_in_top_20_genes"]].describe().T)

                              count         mean          std          min  \
total_counts                27414.0  7698.133789  8000.894531  1001.000000   
n_genes_by_counts           27414.0  1744.139345  1201.347218   201.000000   
pct_counts_mt               27414.0     6.380028     4.551459     0.000000   
pct_counts_ribo             27414.0    24.666594    12.067944     0.613440   
pct_counts_hb               27414.0     0.025634     0.763778     0.000000   
pct_counts_in_top_20_genes  27414.0    30.611475    16.578086     7.138572   

                                    25%          50%           75%  \
total_counts                2492.000000  4276.000000  10015.750000   
n_genes_by_counts            899.250000  1306.000000   2260.750000   
pct_counts_mt                  3.171742     4.909690      8.443174   
pct_counts_ribo               15.952560    23.024910     32.325503   
pct_counts_hb                  0.000000     0.000000      0.003983   
pct_counts_in_top_20_genes    21.

In [5]:
# ============================================================
# 4. Light-touch, composition-aware MAD QC (identical rules to notebook 01)
# ============================================================
def is_outlier(values, nmads):
    v = pd.Series(values).astype(float)
    med = np.median(v); mad = median_abs_deviation(v, nan_policy="omit")
    if mad == 0:
        return pd.Series(False, index=v.index)
    return (v < med - nmads * mad) | (v > med + nmads * mad)

def is_outlier_by_group(ad, metric, nmads, group_col):
    out = pd.Series(False, index=ad.obs_names)
    for g, idx in ad.obs.groupby(group_col, observed=True).groups.items():
        sub = ad.obs.loc[idx, metric]
        if len(sub) < 20:
            continue
        out.loc[idx] = is_outlier(sub, nmads).values
    return out

val.obs["outlier_counts"] = (
    is_outlier(val.obs["log1p_total_counts"], NMADS_COUNTS).values
    | is_outlier(val.obs["log1p_n_genes_by_counts"], NMADS_COUNTS).values
)
val.obs["outlier_top20"] = is_outlier_by_group(
    val, "pct_counts_in_top_20_genes", NMADS_TOP20, celltype_col).values

if MT_RULE == "mad":
    val.obs["mt_outlier"] = is_outlier(val.obs["pct_counts_mt"], 5).values
else:
    val.obs["mt_outlier"] = False
    print("MT rule DISABLED (source matrix already capped at 20% MT).")

val.obs["qc_fail"] = (val.obs["outlier_counts"] | val.obs["outlier_top20"]
                      | val.obs["mt_outlier"])

print("\nPer-rule flags (% of each cell type x tissue group):")
for rule in ["outlier_counts", "outlier_top20", "mt_outlier", "qc_fail"]:
    if not val.obs[rule].any():
        print(f"\n[{rule}] flags nothing."); continue
    tab = (val.obs.groupby([celltype_col, "Class_original"], observed=True)[rule]
           .mean().unstack(fill_value=0) * 100).round(1)
    print(f"\n[{rule}] total = {int(val.obs[rule].sum()):,}")
    print(tab)

n0 = val.n_obs
val = val[~val.obs["qc_fail"]].copy()
print(f"\nQC: {n0:,} -> {val.n_obs:,} ({100 * val.n_obs / n0:.1f}% retained)")

# ---------- differential-loss audit ----------
comp_after = pd.crosstab(val.obs[celltype_col], val.obs["Class_original"]) \
               .reindex(index=comp_before.index, columns=comp_before.columns, fill_value=0)
retained_pct = (comp_after / comp_before.replace(0, np.nan) * 100).round(1)
print("\nCell-type x tissue retention after QC (%):")
print(retained_pct)
print("\nTumor fraction  pre-QC: "
      f"{(comp_before['Border'].sum() + comp_before['Tumor'].sum()) / comp_before.values.sum():.3f}"
      f"  |  post-QC: "
      f"{(comp_after['Border'].sum() + comp_after['Tumor'].sum()) / comp_after.values.sum():.3f}")

MT rule DISABLED (source matrix already capped at 20% MT).

Per-rule flags (% of each cell type x tissue group):

[outlier_counts] flags nothing.

[outlier_top20] total = 586
Class_original    Normal  Border  Tumor
Cell_type                              
B cells              0.0     0.0    0.0
Epithelial cells     9.9     0.5    1.0
Mast cells           1.6     0.0    0.0
Myeloids             1.1     2.9    4.7
Stromal cells        3.6     4.4    3.9
T cells              1.6     0.3    1.6

[mt_outlier] flags nothing.

[qc_fail] total = 586
Class_original    Normal  Border  Tumor
Cell_type                              
B cells              0.0     0.0    0.0
Epithelial cells     9.9     0.5    1.0
Mast cells           1.6     0.0    0.0
Myeloids             1.1     2.9    4.7
Stromal cells        3.6     4.4    3.9
T cells              1.6     0.3    1.6

QC: 27,414 -> 26,828 (97.9% retained)

Cell-type x tissue retention after QC (%):
Class_original    Normal  Border  Tumor
Cell_type 

In [6]:
# ============================================================
# 5. Gene filtering
# ============================================================
print("Before gene filtering:", val.n_vars)
sc.pp.filter_genes(val, min_cells=3)
print("After gene filtering:", val.n_vars)

Before gene filtering: 33694
After gene filtering: 24436


In [7]:
# ============================================================
# 5b. Artifact-gene removal + IEG flagging (identical to notebook 01)
# ============================================================
# Applied here so that Scrublet operates on the SAME gene space as in discovery.
# Notebook 01 removed artifacts BEFORE Scrublet; not mirroring that would make the
# claim of identical cell-intrinsic preprocessing untrue.

HB_GENES  = {"HBA1","HBA2","HBB","HBD","HBG1","HBG2","HBM","HBQ1","HBZ","HBE1"}
SEX_GENES = {"XIST","TSIX","RPS4Y1","RPS4Y2","DDX3Y","UTY","USP9Y",
             "EIF1AY","KDM5D","NLGN4Y","ZFY","TXLNGY","PRKY"}
LNC_GENES = {"MALAT1","NEAT1"}
HSP_KEEP  = {"HSPG2"}

DISSOCIATION_IEG = {
    "FOS","FOSB","JUN","JUNB","JUND","EGR1","EGR2","EGR3","ATF3",
    "IER2","IER3","DUSP1","NR4A1","NR4A2","NR4A3","ZFP36",
    "KLF2","KLF4","KLF6","SOCS3","PPP1R15A","GADD45B",
    "CYR61","CCN1","BTG2","RHOB","PER1","ZNF331","CEBPB","CEBPD",
}

def is_artifact_gene(g):
    g = str(g)
    if g.startswith("MT-"):                                   return True
    if g.startswith(("RPS","RPL","MRPS","MRPL")):             return True
    if re.match(r"^IG[HKL](V|D|J|C|A|G|M|E)", g) or g == "JCHAIN" or g.startswith("IGLL"):
        return True
    if g in HB_GENES:                                         return True
    if (g.startswith("HSP") and g not in HSP_KEEP) or g.startswith("DNAJ"):
        return True
    if g in SEX_GENES:                                        return True
    if g in LNC_GENES:                                        return True
    return False

keep = np.array([not is_artifact_gene(g) for g in val.var_names])
print(f"Artifact genes removed: {int((~keep).sum())} of {val.n_vars}. "
      f"Remaining: {int(keep.sum())}")
print("  HSPH1 removed:", "HSPH1" in val.var_names[~keep].tolist())
val = val[:, keep].copy()

val.var["dissociation_ieg"] = [g in DISSOCIATION_IEG for g in val.var_names]
print("Dissociation IEGs flagged (kept in matrix):",
      int(val.var["dissociation_ieg"].sum()))

Artifact genes removed: 457 of 24436. Remaining: 23979
  HSPH1 removed: True
Dissociation IEGs flagged (kept in matrix): 29


In [8]:
# ============================================================
# 6. Doublet detection (Scrublet, per sample, on raw counts)
# ============================================================
print("Running Scrublet per sample...")
sc.pp.scrublet(val, batch_key=sample_col, random_state=RANDOM_STATE)

print(val.obs["predicted_doublet"].value_counts())
n0 = val.n_obs
val = val[~val.obs["predicted_doublet"]].copy()
print(f"Doublet removal: {n0:,} -> {val.n_obs:,}")

Running Scrublet per sample...
predicted_doublet
False    26767
True        61
Name: count, dtype: int64
Doublet removal: 26,828 -> 26,767


In [9]:
# ============================================================
# 7. Normalize + log1p (identical to notebook 01)
# ============================================================
if "counts" not in val.layers:
    val.layers["counts"] = val.X.copy()

sc.pp.normalize_total(val, target_sum=1e4)
sc.pp.log1p(val)
print("Normalization + log1p complete. X is log-normalized (unscaled). No HVG selection.")
print(val)

Normalization + log1p complete. X is log-normalized (unscaled). No HVG selection.
AnnData object with n_obs × n_vars = 26767 × 23979
    obs: 'Patient', 'Class', 'Sample', 'Cell_type', 'Cell_subtype', 'Class_original', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier_counts', 'outlier_top20', 'mt_outlier', 'qc_fail', 'n_genes', 'doublet_score', 'predicted_doublet'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'dissociation_ieg'
    uns: 'scrublet', 'log1p'
    layers: None (.X), 'counts'


In [10]:
# ============================================================
# 8. Save DL-ready validation object (full gene space)
# ============================================================
val_dl = val.copy()

keep_obs = [c for c in [class_col, "Class_original", celltype_col, patient_col, sample_col,
                        "total_counts", "n_genes_by_counts", "pct_counts_mt"]
            if c in val_dl.obs.columns]
val_dl.obs = val_dl.obs[keep_obs].copy()

val_dl.write_h5ad(OUT_FILE)
print("Saved:", OUT_FILE)
print(val_dl)

print("\nFINAL cell type x tissue composition:")
print(pd.crosstab(val_dl.obs[celltype_col], val_dl.obs["Class_original"]))
print("\nBinary Class:")
print(val_dl.obs[class_col].value_counts())
print("Patients:", val_dl.obs[patient_col].nunique())
print("dissociation_ieg in .var:", "dissociation_ieg" in val_dl.var.columns)

Saved: GSE144735_DL_ready.h5ad
AnnData object with n_obs × n_vars = 26767 × 23979
    obs: 'Class', 'Class_original', 'Cell_type', 'Patient', 'Sample', 'total_counts', 'n_genes_by_counts', 'pct_counts_mt'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'dissociation_ieg'
    uns: 'scrublet', 'log1p'
    layers: None (.X), 'counts'

FINAL cell type x tissue composition:
Class_original    Normal  Border  Tumor
Cell_type                              
B cells             2597    1518    777
Epithelial cells    1027    2793   2183
Mast cells            60     108     76
Myeloids             814     896    885
Stromal cells       3425    1683   2227
T cells             1512    2281   1905

Binary Class:
Class
Tumor     17332
Normal     9435
Name: count, dtype: int64
Patients: 6
dissociation_ieg in .var: True


## Sanity checks before running Option B validation

1. **Cell-type names must match GSE132465** exactly (e.g. `"T cells"`, `"Epithelial cells"`,
   `"B cells"`, `"Myeloids"`, `"Stromal cells"`). If GSE144735 uses different labels,
   add a renaming step in Cell 2 so the one-hot covariate columns align.
2. **Gene symbol convention must match** (same naming as discovery), otherwise the
   intersection in Option B will be small. Print a quick overlap check:
   ```python
   ref = pd.read_csv("gse132465_preprocessing_outputs/GSE132465_HVG3000_gene_list.csv")["gene"]
   print("HVG overlap with validation:", len(set(ref) & set(val_dl.var_names)), "/", len(ref))
   ```
   If overlap is low, the gene IDs differ (e.g. Ensembl vs symbol) and need mapping.
3. `Class` must contain only `Tumor` / `Normal` after the Border merge.


In [11]:
# ============================================================
# 9. Alignment check against the discovery feature space
# ============================================================
KEEP = ["T cells", "Epithelial cells", "B cells", "Myeloids", "Stromal cells"]

print("Validation cell types:", sorted(val_dl.obs[celltype_col].astype(str).unique()))
print("Missing from validation:",
      [c for c in KEEP if c not in val_dl.obs[celltype_col].astype(str).unique()])

n_keep = val_dl.obs[celltype_col].astype(str).isin(KEEP).sum()
print(f"Cells surviving KEEP filter: {n_keep:,}/{val_dl.n_obs:,} "
      f"({100 * n_keep / val_dl.n_obs:.1f}%)")

if os.path.exists(HVG_REF):
    ref = pd.read_csv(HVG_REF)["gene"].tolist()
    inter = sorted(set(ref) & set(val_dl.var_names))
    print(f"\nHVG overlap: {len(inter)}/{len(ref)} ({100 * len(inter) / len(ref):.1f}%)")
    print(f"Notebook 04 will model this INTERSECTION ({len(inter)} genes) — no zero-filling.")
    if len(inter) < 0.8 * len(ref):
        print("WARNING: <80% overlap -> check gene-symbol convention!")
    pd.DataFrame({"gene": inter}).to_csv("shared_feature_space.csv", index=False)
    print("Saved: shared_feature_space.csv")
else:
    print(f"\nDiscovery HVG list not found at {HVG_REF} — run notebook 01 first.")

print("\nClass check:", sorted(val_dl.obs[class_col].astype(str).unique()))
print("Tissue check:", sorted(val_dl.obs['Class_original'].astype(str).unique()))

Validation cell types: ['B cells', 'Epithelial cells', 'Mast cells', 'Myeloids', 'Stromal cells', 'T cells']
Missing from validation: []
Cells surviving KEEP filter: 26,523/26,767 (99.1%)

HVG overlap: 2954/3000 (98.5%)
Notebook 04 will model this INTERSECTION (2954 genes) — no zero-filling.
Saved: shared_feature_space.csv

Class check: ['Normal', 'Tumor']
Tissue check: ['Border', 'Normal', 'Tumor']


## Validation dataset UMAP + data inspection figures

Generate UMAP and inspection figures for the DL-ready GSE144735 validation object. Outputs are written to `validation_dataset_figure/`.

In [ ]:
# ============================================================
# 10. Validation dataset UMAP + data inspection figures
# ============================================================
%run generate_validation_dataset_figures.py
